# BotPsy-0.8B — tool-calling assistant on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/melvin/botpsy-0.8b-tool-calling-511e/colab/BotPsy-0.8B-tool-calling-assistant.ipynb)

[`qvac/BotPsy-0.8B`](https://huggingface.co/qvac/BotPsy-0.8B) is Tether AI Research's 0.8B tool-calling / agentic model (Apache 2.0, ~853M parameters, English recommended). It is a fine-tune of [`Qwen/Qwen3.5-0.8B`](https://huggingface.co/Qwen/Qwen3.5-0.8B): Qwen3.5 architecture (hybrid Gated DeltaNet + gated attention), used **text-only**. The chat template defaults to non-thinking, so this notebook always passes `enable_thinking=True`. Tool calls use the Qwen3.5 XML format (`<tool_call><function=…><parameter=…>`).

**Why this model.** The [card](https://huggingface.co/qvac/BotPsy-0.8B) positions BotPsy as an on-device personal assistant for low-to-medium difficulty tool tasks (calendar, weather, messaging, device stubs, and similar). The authors report that it roughly doubles Qwen3.5-0.8B-Think on BFCL v4 (23.4 → 44.0), ACEBench (30.1 → 58.7), and τ-bench (23.2 → 51.5), and scores 86.1 on BFCL v4's hallucination tests (decline when no suitable tool exists). Those are **self-reported card numbers**; this notebook does not rerun them.

See [Use cases](https://github.com/donvito/notebooks/blob/melvin/botpsy-0.8b-tool-calling-511e/docs/BotPsy-0.8B-tool-calling-assistant.md) for when this 0.8B is a good fit and when it is not.

**Licence:** Apache 2.0. Weights are public — `HF_TOKEN` is optional.

**Runtime:** *Runtime > Change runtime type > T4 GPU*. A free T4 has about 15 GB of VRAM, **no native bfloat16**, and **no FlashAttention 2**. The card loads `dtype=torch.bfloat16`; this notebook loads **float16**. Weights are one **1.7 GB** `model.safetensors` shard. A Colab T4 run peaked at about **1.6 GB** allocated. The card was tested with `transformers` **5.19.0**; this notebook pins **5.20.1** (same Qwen3.5 T4 pin as other notebooks here). Do **not** install `flash-linear-attention` or `causal-conv1d` — building them on a free T4 has killed Colab sessions in this repo. Transformers already has a slower PyTorch fallback for Gated DeltaNet.

**Links:** [model card](https://huggingface.co/qvac/BotPsy-0.8B) · [blog](https://huggingface.co/blog/qvac/botpsy) · [announcement](https://x.com/BitcoinNews/status/2120542632380190952) · [GGUFs](https://huggingface.co/qvac/BotPsy-0.8B-GGUFs) · [Spark-X2.6-1.7B on-device agent](Spark-X2.6-1.7B-on-device-agent.ipynb)


## 1. Check the GPU

Print `torch.cuda.is_available()`, the device name, and free VRAM. This cell imports **torch only** so a later transformers pin can take effect without a restart. T4 (`sm_75`) has no native bfloat16 — **float16** is the path this notebook uses.


In [ ]:
import os

import torch

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Select Runtime > Change runtime type > T4 GPU, then rerun from the top.")

# T4 (sm_75) has no FlashAttention 2. Keep PyTorch SDPA on mem-efficient and math kernels.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

major, minor = torch.cuda.get_device_capability()
try:
    NATIVE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    NATIVE_BF16 = major >= 8

# Card default is bfloat16. T4 is Turing sm_75 — stay on float16.
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
free_b, total_b = torch.cuda.mem_get_info()
print("torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0), f"sm_{major}{minor}", "| dtype:", DTYPE)
print(f"VRAM total: {total_b / 1024**3:.2f} GB | free: {free_b / 1024**3:.2f} GB")
print("native bf16:", NATIVE_BF16, "| FlashAttention-2: off (sdpa / eager / torch fallback)")
if major < 8:
    print("No native bfloat16 on this GPU; loading float16 (1.7 GB shard; T4 peak ~1.6 GB).")


## 2. Install Transformers with Qwen3.5 support

The [model card](https://huggingface.co/qvac/BotPsy-0.8B#transformers) needs a `transformers` release with Qwen3.5 support (tested with **5.19.0**). This notebook pins **5.20.1**, the same pin other Qwen3.5 T4 notebooks in this repo use. The install cell then drops any already-imported `transformers` modules so **Runtime > Run all** can pick up 5.20.1 without *Restart session*. If the version assert still fails, restart once and continue from the GPU cell.

Do not reinstall torch. Do **not** install `flash-attn`, `flash-linear-attention`, or `causal-conv1d`. The card says those two kernels speed up Gated DeltaNet; without them Transformers uses a slower reference implementation, which is the T4-safe path.


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate>=1.1.0" "tokenizers>=0.22" "safetensors>=0.7"

import sys

# Card: transformers with Qwen3.5 (tested 5.19.0). 5.20.1 is this repo's Qwen3.5 T4 pin.
# Do NOT install flash-attn (T4 cannot run FlashAttention 2).
# Do NOT install flash-linear-attention or causal-conv1d: building them
# on ~12.7 GB host RAM has killed a Colab session in this repo. Qwen3.5
# Gated DeltaNet layers use the reference PyTorch path instead.

# Drop a pre-imported Colab transformers so 5.20.1 is visible without Restart session.
for name in list(sys.modules):
    if name == "transformers" or name.startswith("transformers."):
        del sys.modules[name]

import transformers
from packaging.version import Version

assert Version(transformers.__version__.split("+")[0]) >= Version("5.19.0"), (
    f"transformers {transformers.__version__} is loaded; BotPsy-0.8B needs 5.19.0+ "
    "(Qwen3.5). Runtime > Restart session, then rerun from the GPU check cell."
)
print("transformers", transformers.__version__)


## 3. Load BotPsy-0.8B

`AutoTokenizer` + `AutoModelForCausalLM`, as on the [Transformers snippet](https://huggingface.co/qvac/BotPsy-0.8B#transformers). No `trust_remote_code`: the card says Transformers, vLLM, and llama.cpp run BotPsy without custom code. The saved architecture is `Qwen3_5ForConditionalGeneration` (Qwen3.5 hybrid, used text-only); if this build refuses CausalLM, the cell falls back to `AutoModelForImageTextToText` and still generates from token ids. Attention is **sdpa**, then **eager**. The first run downloads about 1.7 GB. A Colab T4 run peaked at about **1.6 GB** allocated.

Sampling follows `generation_config.json` and the card: `do_sample=True`, `temperature=0.3`, `top_p=0.95`, `top_k=20`. Evaluations also used `presence_penalty=1.5` (the card says that is available in vLLM and llama.cpp); this Transformers path sticks to the saved config. The card snippet uses `max_new_tokens=4096` so thinking-mode answers are not cut off mid-sentence — same budget here, well below the 32,768 eval cap.


In [ ]:
import gc
import json
import re

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "qvac/BotPsy-0.8B"


def print_vram(label=""):
    suffix = f" ({label})" if label else ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    free_now, total_now = torch.cuda.mem_get_info()
    print(
        f"VRAM{suffix}: allocated {allocated:.2f} GB | reserved {reserved:.2f} GB | "
        f"peak {peak:.2f} GB | free {free_now / 1024**3:.2f}/{total_now / 1024**3:.2f} GB"
    )


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


def load_botpsy(attn):
    kwargs = {
        "dtype": DTYPE,
        "device_map": "auto",
        "attn_implementation": attn,
    }
    try:
        return AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs).eval()
    except (ValueError, OSError) as exc:
        print("AutoModelForCausalLM refused this checkpoint:", exc)
        from transformers import AutoModelForImageTextToText

        print("Falling back to AutoModelForImageTextToText (text-only generate).")
        return AutoModelForImageTextToText.from_pretrained(MODEL_ID, **kwargs).eval()


model = None
ATTN = None
last_error = None
for attn in ("sdpa", "eager"):
    try:
        model = load_botpsy(attn)
        ATTN = attn
        break
    except Exception as exc:
        last_error = exc
        print(f"{attn} load failed:", exc)
        del model
        model = None
        torch.cuda.empty_cache()

if model is None:
    raise RuntimeError(f"Could not load {MODEL_ID}") from last_error

n_params = sum(p.numel() for p in model.parameters())
print("parameters:", f"{n_params / 1e6:.0f}M")
print("dtype:", next(model.parameters()).dtype)
print("attn:", ATTN)
print_vram("after load")

THINK_CLOSE = "</think>"


def split_think(text, *, thinking_enabled=True):
    # Card chat template: enable_thinking=True prepends <think> on the
    # generation prompt. New tokens are the think channel until </think>.
    if THINK_CLOSE in text:
        think, answer = text.split(THINK_CLOSE, 1)
        return think.replace("<think>", "").strip(), answer.strip()
    if thinking_enabled or text.lstrip().startswith("<think>"):
        return text.replace("<think>", "").strip(), ""
    return "", text.strip()


def parse_tool_calls(text):
    # Qwen3.5 XML format from the card:
    # <tool_call>
    # <function=create_event>
    # <parameter=title>
    # Dentist
    # </parameter>
    # ...
    # </function>
    # </tool_call>
    calls = []
    for match in re.finditer(
        r"<tool_call>\s*<function=([^>]+)>(.*?)</function>\s*</tool_call>",
        text,
        flags=re.DOTALL,
    ):
        name = match.group(1).strip()
        args = {}
        for pname, pvalue in re.findall(
            r"<parameter=([^>]+)>\s*(.*?)\s*</parameter>",
            match.group(2),
            flags=re.DOTALL,
        ):
            raw_value = pvalue.strip()
            try:
                args[pname] = json.loads(raw_value)
            except json.JSONDecodeError:
                args[pname] = raw_value
        calls.append({"name": name, "arguments": args})
    return calls


def generate_ids(inputs, max_new_tokens, *, seed=None):
    # Card Transformers snippet + generation_config.json.
    # Optional seed keeps a multi-step wrap-up from drifting off the tool JSON.
    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    with torch.inference_mode():
        return model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.95,
            top_k=20,
        )


def chat(messages, *, tools=None, max_new_tokens=4096, seed=None):
    template = {
        "add_generation_prompt": True,
        "enable_thinking": True,
        "return_dict": True,
        "return_tensors": "pt",
    }
    if tools is not None:
        template["tools"] = tools
    inputs = tokenizer.apply_chat_template(messages, **template)
    device = next(model.parameters()).device
    if hasattr(inputs, "to"):
        inputs = inputs.to(device)
    else:
        inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
    generated = generate_ids(inputs, max_new_tokens, seed=seed)
    prompt_len = inputs["input_ids"].shape[-1]
    text = tokenizer.decode(generated[0, prompt_len:], skip_special_tokens=True)
    print_vram()
    return text


def run_tool(name, arguments, functions):
    fn = functions.get(name)
    if fn is None:
        return {"error": f"unknown tool: {name}"}
    try:
        return fn(**arguments)
    except TypeError as exc:
        return {"error": str(exc)}


def run_tool_loop(question, tools, functions, *, max_rounds=4, max_new_tokens=4096, seed=None):
    messages = [{"role": "user", "content": question}]
    tool_results = []
    print(f"[user] {question}")
    for round_number in range(1, max_rounds + 1):
        raw = chat(messages, tools=tools, max_new_tokens=max_new_tokens, seed=seed)
        think, rest = split_think(raw, thinking_enabled=True)
        calls = parse_tool_calls(raw)
        print(f"\n=== model (round {round_number}) ===")
        print("----- think -----")
        print(think or "(empty think channel)")
        print("----- response -----")
        print(rest or "(no response channel)")
        print("parsed calls:", calls)
        if not calls:
            print("\n--- final answer ---")
            print(rest or raw)
            return {
                "raw": raw,
                "think": think,
                "answer": rest,
                "calls": [],
                "tool_results": tool_results,
            }
        content = re.sub(r"<tool_call>.*?</tool_call>", "", rest, flags=re.DOTALL).strip()
        assistant = {
            "role": "assistant",
            "content": content,
            "tool_calls": [
                {"function": {"name": call["name"], "arguments": call["arguments"]}}
                for call in calls
            ],
        }
        if think:
            assistant["reasoning_content"] = think
        messages.append(assistant)
        for call in calls:
            payload = run_tool(call["name"], call["arguments"], functions)
            tool_results.append(
                {"name": call["name"], "arguments": call["arguments"], "result": payload}
            )
            print(f"[tool] {call['name']}({call['arguments']}) -> {json.dumps(payload)}")
            messages.append(
                {"role": "tool", "content": json.dumps(payload, separators=(",", ":"))}
            )
    print(f"\nNo final answer after {max_rounds} tool-calling rounds.")
    return {
        "raw": raw,
        "think": think,
        "answer": rest,
        "calls": calls,
        "tool_results": tool_results,
    }


## 4. Demo A — one calendar tool call

Small tools schema: `create_event` only, same shape as the [card example](https://huggingface.co/qvac/BotPsy-0.8B#transformers). The stub is a fake Python function over an in-memory sample calendar (not a live API). The notebook parses the Qwen3.5 `<tool_call>` block, runs the stub, feeds the result back as role `tool`, and asks for a final answer. The prompt names a specific start time so the model does not have to guess a time of day.


In [ ]:
CREATE_EVENT_TOOL = {
    "type": "function",
    "function": {
        "name": "create_event",
        "description": "Add an event to the user's sample calendar",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "day": {"type": "string", "description": "Day of the week"},
                "time": {"type": "string", "description": "Start time, 24h HH:MM"},
            },
            "required": ["title", "day", "time"],
        },
    },
}

SAMPLE_CALENDAR = [
    {"title": "Team standup (sample)", "day": "Monday", "time": "09:30"},
    {"title": "Design critique (sample)", "day": "Wednesday", "time": "11:00"},
    {"title": "Ferry to Cascais (sample)", "day": "Saturday", "time": "10:00"},
]


def create_event(title: str, day: str, time: str) -> dict:
    event = {
        "title": title,
        "day": day,
        "time": time,
        "calendar": "sample-calendar",
        "status": "added",
        "note": "sample data, not a live calendar",
    }
    SAMPLE_CALENDAR.append(event)
    return event


print("=== Demo A: single create_event call ===")
run_tool_loop(
    "Add the dentist on Friday at 15:00 to my sample calendar.",
    tools=[CREATE_EVENT_TOOL],
    functions={"create_event": create_event},
    max_rounds=2,
)


## 5. Demo B — multi-step weather + calendar

Three tools: `get_weather`, `list_events`, and `create_event`. Saturday already has a sample 10:00 ferry, so the calendar JSON is non-empty and includes an explicit `day_status` (`has_events` / `free`). The prompt asks the model to **quote those tool fields**, not to answer a yes/no “was it busy?” question — that wording produced a contradiction on a T4 run (`busy: Yes` while `events` was empty). Weather and calendar rows are hardcoded so the loop stays offline and does not assume the current time of day. Demo B uses the card sampling settings with seed `0` so the wrap-up stays on the tool JSON.


In [ ]:
GET_WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get a sample weather snapshot for a city. Not a live forecast.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. Lisbon"},
            },
            "required": ["city"],
        },
    },
}

LIST_EVENTS_TOOL = {
    "type": "function",
    "function": {
        "name": "list_events",
        "description": (
            "List events on the sample calendar for one day of the week. "
            "Returns events, count, and day_status: 'has_events' or 'free'."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "day": {"type": "string", "description": "Day of the week"},
            },
            "required": ["day"],
        },
    },
}

MULTI_TOOLS = [GET_WEATHER_TOOL, LIST_EVENTS_TOOL, CREATE_EVENT_TOOL]


def get_weather(city: str) -> dict:
    catalog = {
        "lisbon": {
            "city": "Lisbon",
            "celsius": 18,
            "condition": "light rain",
            "note": "sample catalog, not a live forecast",
        },
        "singapore": {
            "city": "Singapore",
            "celsius": 31,
            "condition": "partly cloudy",
            "note": "sample catalog, not a live forecast",
        },
    }
    hit = catalog.get(city.strip().lower())
    if hit is None:
        return {
            "city": city,
            "celsius": 21,
            "condition": "clear",
            "note": "sample default, not a live forecast",
        }
    return hit


def list_events(day: str) -> dict:
    rows = [
        {
            "title": item["title"],
            "day": item["day"],
            "time": item["time"],
        }
        for item in SAMPLE_CALENDAR
        if item["day"].strip().lower() == day.strip().lower()
    ]
    return {
        "day": day,
        "events": rows,
        "count": len(rows),
        "day_status": "has_events" if rows else "free",
        "note": "sample calendar, not a live account",
    }


print("=== Demo B: weather + list_events + create_event ===")
DEMO_B = run_tool_loop(
    (
        "I will be in Lisbon on Saturday. Check the sample weather for Lisbon, "
        "then list Saturday on my sample calendar. If the weather condition is "
        "rain, add an indoor event titled 'LX Museum (sample)' at 14:00. After "
        "the tools return, write three short lines that quote the tool JSON "
        "only: (1) Lisbon condition and celsius, (2) Saturday day_status plus "
        "each event title and 24-hour time, (3) the create_event title and "
        "time if you added one. Do not answer yes or no. Do not use the word "
        "busy. Do not invent events."
    ),
    tools=MULTI_TOOLS,
    functions={
        "get_weather": get_weather,
        "list_events": list_events,
        "create_event": create_event,
    },
    max_rounds=4,
    seed=0,
)


The next cell checks that the Demo B wrap-up quotes the tool JSON instead of contradicting it (the failure mode was `busy: Yes` on an empty Saturday list). It prints OK / MISS; it does not raise.

In [ ]:
print("=== Demo B: tool-result check ===")
answer = (DEMO_B.get("answer") or DEMO_B.get("raw") or "").lower()
results = DEMO_B.get("tool_results") or []


def last_tool(name):
    hits = [row["result"] for row in results if row["name"] == name]
    return hits[-1] if hits else None


weather = last_tool("get_weather")
listed = last_tool("list_events")
created = last_tool("create_event")
checks = []

if weather:
    condition = str(weather.get("condition", "")).lower()
    celsius = str(weather.get("celsius", ""))
    checks.append(
        (
            "quotes Lisbon condition or celsius",
            bool(condition and condition in answer) or bool(celsius and celsius in answer),
        )
    )
else:
    checks.append(("called get_weather", False))

if listed:
    status = str(listed.get("day_status", "")).lower()
    events = listed.get("events") or []
    titles = [str(item.get("title", "")).lower() for item in events]
    times = [str(item.get("time", "")) for item in events]
    quoted_status = bool(status) and status in answer
    quoted_event = any(title.split("(")[0].strip() in answer for title in titles if title) or any(
        time in answer for time in times if time
    )
    empty = len(events) == 0
    says_busy_yes = "busy" in answer and "yes" in answer
    says_no_events = "no event" in answer or "no events" in answer
    contradicted = (empty and says_busy_yes) or (not empty and says_no_events)
    checks.append(("quotes day_status or a Saturday title/time", quoted_status or quoted_event))
    checks.append(("does not contradict list_events", not contradicted))
else:
    checks.append(("called list_events", False))

if created:
    title = str(created.get("title", "")).lower()
    time = str(created.get("time", ""))
    checks.append(
        (
            "quotes create_event title or time",
            (bool(title) and title.split("(")[0].strip() in answer)
            or (bool(time) and time in answer),
        )
    )
elif weather and "rain" in str(weather.get("condition", "")).lower():
    checks.append(("called create_event after rain", False))

for label, ok in checks:
    print(("OK   " if ok else "MISS ") + label)

if checks and all(ok for _, ok in checks):
    print("Demo B wrap-up agrees with the tool JSON.")
else:
    print("Demo B wrap-up does not fully match the tool JSON; read the final answer above.")

## 6. Demo C — no matching tool (decline, do not invent a call)

Same weather + calendar tools as above. The request is a messaging / bank-transfer action those tools cannot serve. The card highlights this failure mode on BFCL v4's hallucination tests (self-reported **86.1**, highest of the models they tested): decline when no appropriate tool exists, rather than inventing a call. A text answer with **no** `<tool_call>` block is the hit; a fabricated `send_message` / `transfer_money` name is a miss.


In [ ]:
print("=== Demo C: request with no suitable tool ===")
result = run_tool_loop(
    (
        "Please send 40 euros from my sample checking account to the sample "
        "landlord for this week's rent, and text Alex that the transfer went through."
    ),
    tools=[GET_WEATHER_TOOL, CREATE_EVENT_TOOL],
    functions={"get_weather": get_weather, "create_event": create_event},
    max_rounds=2,
)
if result["calls"]:
    print(
        "\nNote: the model emitted a tool call even though weather/calendar "
        "cannot send money or messages. That is a hallucinated call, not a crash."
    )
else:
    print("\nNo <tool_call> block — the model answered in text instead of inventing a tool.")


## 7. Caveats and cleanup

- **Thinking on.** BotPsy was trained and evaluated in thinking mode. The chat template defaults to non-thinking, so every generate here passes `enable_thinking=True`.
- **Sampling.** `generation_config.json` / card Transformers path: `temperature=0.3`, `top_p=0.95`, `top_k=20`, `do_sample=True`. Demo B also passes `seed=0` so the wrap-up is less likely to drift off the tool JSON. Evaluations also used `presence_penalty=1.5` in vLLM and llama.cpp; this notebook does not set that on `model.generate`.
- **Token budget.** Card snippet: `max_new_tokens=4096`. Evals allowed up to 32,768. A 16 GB T4 will not hold that KV cache on top of the weights; 4096 is the demo cap so thinking traces are less likely to be cut off mid-sentence.
- **Float16 on a T4.** Saved dtype is BF16. T4 has no native bfloat16, so this notebook uses float16 (1.7 GB safetensors shard). A Colab T4 run peaked at about **1.6 GB** allocated. Newer GPUs (L4, A100) can load bfloat16, the card default.
- **No FLA / causal-conv1d.** The card's fast Gated DeltaNet kernels are optional. This notebook uses the Transformers PyTorch fallback so a free T4 does not compile extra CUDA extensions.
- **Text-only.** The Qwen3.5 architecture can carry a vision config; BotPsy's card says it is used text-only. There is no image demo here.
- **Low-to-medium difficulty.** The card: best suited to tasks of low to medium difficulty; English recommended; not for safety-critical or high-stakes automated decisions, or for applications involving money, personal information, health, and other sensitive topics. Demo C's rent transfer is a **refusal** test, not a payments demo.
- **Self-reported scores.** BFCL / ACEBench / τ-bench / hallucination numbers in the intro are from the [model card](https://huggingface.co/qvac/BotPsy-0.8B). This notebook does not rerun them.
- **Optional GGUF.** [`qvac/BotPsy-0.8B-GGUFs`](https://huggingface.co/qvac/BotPsy-0.8B-GGUFs) has BF16, Q8_0, and Q4_K_M for llama.cpp and the QVAC SDK. Prefer the Transformers float16 path above on this T4 if it fits.
- **Demo B wrap-up.** Saturday is a non-empty sample day (`Ferry to Cascais (sample)` at 10:00) with an explicit `day_status` field. The prompt asks the model to quote tool fields, not to answer a yes/no “busy” question. A check cell prints whether the final answer agrees with the tool JSON.
- **Related notebook.** [`Spark-X2.6-1.7B-on-device-agent.ipynb`](Spark-X2.6-1.7B-on-device-agent.ipynb) is another small on-device agent with a weather/time tool loop (SparkLLM, ~1.71B).

The next cell drops the model so you can load something else on the same T4.


In [ ]:
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print_vram("after cleanup")
